# k-Nearest Neighbours for CIFAR-10 Image Classification

## What this notebook demonstrates

- CIFAR-10 image classification using raw pixels
- k-nearest neighbours classification
- L1 and L2 distance computation with loop-based and vectorized implementations
- cross-validation over the number of neighbours
- qualitative nearest-neighbour retrieval analysis

## Academic context

This notebook is a cleaned portfolio version of MSc coursework and implementation practice. It is included to demonstrate foundational computer vision and deep learning skills.

## Local helper package

This notebook uses the included project-local `engine/` package for coursework helper code such as data loading, classifiers, layers, optimization, and visualization utilities.


In [ ]:
# Project-local setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "engine").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the project root containing the engine/ package.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


### Run setup code


In [ ]:
# Run some setup code for this notebook.

import random
import numpy as np
import matplotlib.pyplot as plt

# This is a bit of magic to make matplotlib figures appear inline in the notebook
# rather than in a new window.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Reload local helper modules during iterative notebook work.
%load_ext autoreload
%autoreload 2


### CIFAR-10 Data Loading and Pre-processing


In [ ]:
from engine.data_utils import load_CIFAR10

# Load the raw CIFAR-10 data.
cifar10_dir = str(DATA_DIR / 'cifar-10-batches-py')

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


In [ ]:
# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
classes = ['Plane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls[:7])
plt.show()


Due to computation limitations, we choose to subsample the data for more efficient code execution. We choose to keep only ``5000`` images for training and ``500`` images for testing. We also reshape the ``3-dimensional`` *tensor* data to ``1-dimensional`` *vector* data. This is a preparation step for the k-NN classification we are about to run below.


In [ ]:
# Subsample the data for more efficient code execution in this notebook
num_training = 5000
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]

num_test = 500
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]

# Convert arrays uint8 -> float
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# Reshape the image data into rows
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
print(X_train.shape, X_test.shape)


You can save these arrays (X_train, y_train, X_test, y_test) in a file format called ``.npy`` (numpy array), so that in case your runtime crashes, you can easily load them again WITHOUT having to load the whole dataset again. Let's have a look on how to do that!


In [ ]:
# First, let's create a new folder called arrays
# inside the data directory. Specify dir:
data_dir = str(DATA_DIR)
arrays_dir = os.path.join(data_dir, 'arrays')

# Create arrays directory
if not os.path.exists(arrays_dir):
    os.makedirs(arrays_dir)

# Save arrays in .npy files inside the new arrays dir
np.save(os.path.join(arrays_dir, 'X_train_knn.npy'), X_train)
np.save(os.path.join(arrays_dir, 'y_train_knn.npy'), y_train)
np.save(os.path.join(arrays_dir, 'X_test_knn.npy'), X_test)
np.save(os.path.join(arrays_dir, 'y_test_knn.npy'), y_test)

# Check that arrays are indeed saved
os.listdir(arrays_dir)


### Load .npy files


Now, you can use the following coding snippet in case the runtime crashes to easily load these arrays.


In [ ]:
# Specify data directory again, so that arrays dir is
# automatically retrieved and arrays can be loaded from it
data_dir = str(DATA_DIR)
arrays_dir = os.path.join(data_dir, 'arrays')

# Load arrays if needed. Uncomment to use!
X_train = np.load(os.path.join(arrays_dir, 'X_train_knn.npy'))
y_train = np.load(os.path.join(arrays_dir, 'y_train_knn.npy'))
X_test = np.load(os.path.join(arrays_dir, 'X_test_knn.npy'))
y_test = np.load(os.path.join(arrays_dir, 'y_test_knn.npy'))

# Num of samples for training and test
num_training = X_train.shape[0]
num_test = X_test.shape[0]


### Train k-NN classifier, Compute distance (two loops)


The kNN classifier consists of two stages:

- During training, the classifier takes the training data and simply remembers it
- During testing, kNN classifies every test image by comparing to all training images and transfering the labels of the k most similar training examples
- The value of k is cross-validated


In [ ]:
from engine.classifiers.k_nearest_neighbor import KNearestNeighbor

# Create a kNN classifier instance.
# Remember that training a kNN classifier is a noop:
# the Classifier simply remembers the data and does no further processing
classifier = KNearestNeighbor()
classifier.train(X_train, y_train)


We would now like to classify the test data with the kNN classifier. Recall that we can break down this process into two steps:

1. First we must compute the distances between all test examples and all train examples.
2. Given these distances, for each test example we find the k nearest examples and have them vote for the label

Lets begin with computing the distance matrix between all training and test examples. For example, if there are ``Ntr`` training examples and ``Nte`` test examples, this stage should result in a ``Nte x Ntr`` matrix where each element ``(i,j)`` is the distance between the ``i-th`` test and ``j-th`` train example.

**Note: For the three distance computations that this notebook implements in this notebook, you may not use the np.linalg.norm() function that numpy provides.**

**Implementation note.**

This section validates the function `compute_distances_two_loops` that uses a (very inefficient) double loop over all pairs of (test, train) examples and computes the distance matrix one element at a time.


In [ ]:
# compute_distances_two_loops.

# Validate the implementation:
dists_two = classifier.compute_distances_two_loops(X_test)
print(dists_two.shape)


In [ ]:
# We can also visualize the distance matrix: each row is a single test example and
# its distances to training examples

f = plt.figure(figsize=(24,4))
plt.imshow(dists_two, interpolation='none')


**Concept check.**

Notice the structured patterns in the distance matrix, where some rows or columns are visible *brighter/darker*. (Note that with the default color scheme *black* indicates *low distances* while *white* indicates *high distances*.)

- What in the data is the cause behind these distinctly bright/dark rows?
- What in the data is the cause behind these distinctly bright/dark columns?

Before answering, let us recall what this distance matrix refers to. ``dists[i, j]`` corresponds to the L2 distance between the ``ith`` test and ``jth`` training point.

**Answer.** The i-th row contains the distances of the i-th test sample with the training samples. A white row would indicate that the i-th test sample has a high L2 distance from most of the training samples, while a black row would indicate the opposite. The same applies to the columns, where the j-th column contains the distances of the j-th training sample from all the test samples.

Thus, the distinctive brightness of the rows and columns is due to the fact that some of the testing feature vectors lie closer to some of the training feature vectors in the feature space (in R^3072) and further away from some others. The training feature vectors that are closer (k of them) will determine the class of the testing sample.

Considering the simplistic representation that we've used for each image (a flattened vector of pixel values), one should not expect to have a rich enough representation or a distinctive enough feature space and thus a similarity classifier can only do so much.


### Predict Labels


**Implementation note.**

This section uses and validates the implementation of `predict_labels`. This function should take as input the distance matrix between test and training points and output a label for each test point.


In [ ]:
# We use k = 1 (which is Nearest Neighbor).
y_test_pred = classifier.predict_labels(dists_two, k=1)

# Compute and print the fraction of correctly predicted examples
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))


Expected result: approximately `27%` accuracy (if more, then even better). Now lets try out a larger `k`, say `k = 5`:


In [ ]:
y_test_pred = classifier.predict_labels(dists_two, k=5)
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))


Expected result: a slightly better performance than with `k = 1`. If not, it's still fine!


**Concept check.**

Apart from L2 distance, we can also use other distance metrics such as L1 distance.
For pixel values $p_{ij}^{(k)}$ at location $(i,j)$ of some image $I_k$,
the mean $\mu$ across all pixels over all images is $$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$

The general standard deviation $\sigma$ is defined similarly.

Which of the following preprocessing steps will *not* change the performance of a k-Nearest Neighbor classifier that uses L1 distance? Select **all** that apply.
1. Subtracting the mean $\mu$ ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$).
2. Subtracting the mean $\mu$ and dividing by the standard deviation $\sigma$.
3. None of the above

**Answer.** 3


**Explanation.** Centering the data around 0 by subtracting the mean will change the relative distances of points if we use L1. If we also divide by the std this will also impact the relative distances of points by scaling them. Thus, in both cases the performance of kNN will change.


### Compute distance (one loop)


**Implementation note.**

This section uses and validates the implementation of `compute_distances_one_loop` that uses partial vectorization with one loop to speed up the distance matrix computation.


In [ ]:
# Now lets speed up distance matrix computation by using partial vectorization
# with one loop. Implement the function compute_distances_one_loop and run the
# code below:
dists_one = classifier.compute_distances_one_loop(X_test)

# To ensure that our vectorized implementation is correct, we make sure that it
# agrees with the naive implementation. There are many ways to decide whether
# two matrices are similar; one of the simplest is the Frobenius norm. In case
# you haven't seen it before, the Frobenius norm of two matrices is the square
# root of the squared sum of differences of all elements; in other words, reshape
# the matrices into vectors and compute the Euclidean distance between them.
difference = np.linalg.norm(dists_two - dists_one, ord='fro')
print('One loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')


### Compute distance (no loops)


**Implementation note.**

This section uses the implementation of `compute_distances_no_loops` that uses the fully vectorized version.


In [ ]:
# and run the code
dists = classifier.compute_distances_no_loops(X_test)

# check that the distance matrix agrees with the one we computed before:
difference = np.linalg.norm(dists_two - dists, ord='fro')
print('No loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')


In [ ]:
# Let's compare how fast the implementations are
def time_function(f, *args):
    """
    Call a function f with args and return the time (in seconds) that it took to execute.
    """
    import time
    tic = time.time()
    f(*args)
    toc = time.time()
    return toc - tic

two_loop_time = time_function(classifier.compute_distances_two_loops, X_test)
print('Two loop version took %f seconds' % two_loop_time)

one_loop_time = time_function(classifier.compute_distances_one_loop, X_test)
print('One loop version took %f seconds' % one_loop_time)

no_loop_time = time_function(classifier.compute_distances_no_loops, X_test)
print('No loop version took %f seconds' % no_loop_time)

# Expected: see significantly faster performance with the fully vectorized implementation!

# NOTE: depending on what machine you're using,
# you might not see a speedup when you go from two loops to one loop.
# Also, if a slow-down is observed from one loop to no loops, that's fine.


### k-fold cross validation


#### Cross-validation

We have implemented the k-Nearest Neighbor classifier but we set the value ``k = 5`` arbitrarily. We will now determine the best value of this hyperparameter with cross-validation.

**Implementation note.**

1. Split up the training data into folds.
2. Perform k-fold cross validation to find the best value of ``k``.


In [ ]:
num_folds = 5
k_choices = [1, 3, 5, 8, 10, 12, 15, 20, 50, 100]

X_train_folds = []
y_train_folds = []
# Split up the training data into folds. After splitting, X_train_folds and    #
# y_train_folds should each be lists of length num_folds, where                #
# y_train_folds[i] is the label vector for the points in X_train_folds[i].     #
# Hint: Look up the numpy array_split function.                                #
X_train_folds = np.array_split(X_train, num_folds)
y_train_folds = np.array_split(y_train, num_folds)

# A dictionary holding the accuracies for different values of k that we find
# when running cross-validation. After running cross-validation,
# k_to_accuracies[k] should be a list of length num_folds giving the different
# accuracy values that we found when using that value of k.
k_to_accuracies = {}


# Perform k-fold cross validation to find the best value of k. For each        #
# possible value of k, run the k-nearest-neighbor algorithm num_folds times,   #
# where in each case you use all but one of the folds as training data and the #
# last fold as a validation set. Store the accuracies for all fold and all     #
# values of k in the k_to_accuracies dictionary.                               #
# Perform k-fold cross-validation for each k
for k in k_choices:

    k_to_accuracies[k] = []
    for i in range(num_folds):

        # Create the training set by combining all folds except the i-th one
        X_train_fold = np.concatenate([X_train_folds[j] for j in range(num_folds) if j != i], axis=0)
        y_train_fold = np.concatenate([y_train_folds[j] for j in range(num_folds) if j != i], axis=0)

        X_val = X_train_folds[i]
        y_val = y_train_folds[i]

        classifier.train(X_train_fold, y_train_fold)

        dists = classifier.compute_distances_no_loops(X_val)

        # Predict
        y_pred = classifier.predict_labels(dists, k)

        # Calculate accuracy (the way it was calculated earlier) and store it
        num_val = y_val.shape[0]
        num_correct = np.sum(y_pred == y_val)
        accuracy = float(num_correct) / num_val

        k_to_accuracies[k].append(accuracy)



# Print out the computed accuracies
for k in sorted(k_to_accuracies):
    for accuracy in k_to_accuracies[k]:
        print('k = %d, accuracy = %f' % (k, accuracy))


In [ ]:
# plot the raw observations
for k in k_choices:
    accuracies = k_to_accuracies[k]
    plt.scatter([k] * len(accuracies), accuracies)

# plot the trend line with error bars that correspond to standard deviation
accuracies_mean = np.array([np.mean(v) for k,v in sorted(k_to_accuracies.items())])
accuracies_std = np.array([np.std(v) for k,v in sorted(k_to_accuracies.items())])
plt.errorbar(k_choices, accuracies_mean, yerr=accuracies_std)
plt.title('Cross-validation on k')
plt.xlabel('k')
plt.ylabel('Cross-validation accuracy')
plt.show()


In [ ]:
# calculate the mean accuracy for each k and choose the best
for k in sorted(k_to_accuracies):
    mean = np.mean(k_to_accuracies[k])
    print('k = %d, mean accuracy = %f' % (k, mean))


In [ ]:
# Based on the cross-validation results above, choose the best value for k,
# retrain the classifier using all the training data, and test it on the test
# data. Expected: be able to get above 28% accuracy on the test data.
best_k = 10

classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
y_test_pred = classifier.predict(X_test, k=best_k)

# Compute and display the accuracy
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))


**Concept check.**

Which of the following statements about $k$-Nearest Neighbor ($k$-NN) are true in a classification setting, and for all $k$? Select all that apply.
1. The decision boundary of the k-NN classifier is linear.
2. The training error of a 1-NN will always be lower than or equal to that of 5-NN.
3. The test error of a 1-NN will always be lower than that of a 5-NN.
4. The time needed to classify a test example with the k-NN classifier grows with the size of the training set.
5. None of the above.

**Answer.** 2, 4


**Explanation.** 1) It depends on the value of k and generally it is not linear. 2) Noise and outliers influence dramatically L2 distance and thus k=5 can cause kNN to misclassify, while k=1 will classify according to the closest neighbors class. 3) Smaller values of k tend to overfit and prevent generalization, thus in a test set a higher value of k will generalize better. 4) Yes because for each test point, you need to compute the distance to all the points in the training set to find the k nearest neighbors.


**Implementation note.**

This section randomly select one image from each of the 10 classes in the CIFAR-10 test set. Using the k-Nearest Neighbor classifier, you will find and display the top-10 closest images from the training set for each selected image. The following conditions will apply:

- For each selected test image, display the 10 closest images based on their distances in the feature space.
- Below each image, display its predicted class.
- Apply a green border if the retrieved image belongs to the same class as the test image (i.e., correctly retrieved), and a red border otherwise (i.e., incorrectly retrieved).
  
This task allows us to visually inspect how well the k-NN classifier performs in "retrieving" images.


In [ ]:
# Visualize the top-10 closest images to a randomly chosen image from each     #
# class in the CIFAR-10 test set. For each class, select one random test image #
# and find the 10 closest training images. Use the distances to color-code     #
# the retrieved images with a green border for correct labels and red for      #
# incorrect ones.                                                              #
import matplotlib.patches as patches

classes = np.unique(y_train)

test_neighbors_dict = {}

# get neighbors
for i, class_id in enumerate(classes):
    class_indices = np.where(y_test == class_id)[0]
    random_idx = np.random.choice(class_indices)

    test_image = X_test[random_idx]
    test_label = y_test[random_idx]

    # Find the 10 nearest neighbors
    _, indices = classifier.kneighbors(test_image.reshape(1, -1), k=10)

    test_neighbors_dict[random_idx] = indices

# plot results
rows = 11
cols = 10
fig, axes = plt.subplots(rows, cols, figsize=(15, 15))

for i, (test_idx, neighbors) in enumerate(test_neighbors_dict.items()):
    # Plot the test image in the first row
    ax = axes[0, i]
    ax.imshow(X_test[test_idx].reshape((32, 32, 3)).astype('uint8'))
    ax.axis('off')
    ax.set_title(f'Class {y_test[test_idx]}')

    # Plot the nearest neighbors
    for j, neighbor_idx in enumerate(neighbors):
        ax = axes[j+1, i]
        ax.imshow(X_train[neighbor_idx].reshape((32, 32, 3)).astype('uint8'))
        ax.axis('off')

        # Check if the label of the neighbor is the same as the test samples label
        if y_train[neighbor_idx] == y_test[test_idx]:
            border_color = (0.5, 1, 0.5)
        else:
            border_color = 'red'

        border = patches.Rectangle((0, 0), 1, 1, linewidth=3, edgecolor=border_color, facecolor='none', transform=ax.transAxes)
        ax.add_patch(border)

plt.tight_layout()
plt.show()


**Concept check.**

In the previous visualization, you examined how well k-NN using L2 distance “retrieves” the top-10 closest images to a randomly selected image from each class. In a k-NN classification scenario where k=10, the algorithm would assign the test image to the class that appears most frequently among the retrieved images.

Take a moment to qualitatively evaluate the results:

•	When does retrieval fail to find images from the correct class?
  -	Are there cases where the retrieved images look similar morphologically (e.g. shapes) but do not belong to the same class (e.g., different animals with similar features)?

•	When does retrieval succeed?
  -	Are there cases where the retrieved images belong to the same class as the test image? Are the retrieved images only similar because they appear morphologically alike (e.g., airplanes in similar poses), rather than being semantically related?

Reflect on the following:

•	In this notebook, L2 distance is used to measure similarity based on pixel values. Why might this approach fail to capture the true semantic similarity between images?

•	How does relying on pixel values cause L2 distance to prioritize superficial details over the actual content of the images (e.g., focusing on color or texture rather than object type)?

**Answer.** In our representation each feature vector depends solely on pixel values and nothing else. Such a representation does not capture the distinctive aspects of each object that the image depicts, such as geometric shape, features like corners or edges, correlations between  pixels etc. As a result the feature space is not guaranteed to bring images of the same object closer. There are many aspects that can influence such a poor representation like different illuminations, camera angle, occlusion, pose etc.

Thus, correct/false classification for two images depends only on whether the corresponding pixel values (at the same position) are similar/different which constributes to the l2 distance of the vectors. The more the corresponding pixel values diverge the bigger the distance.


**Implementation note.**

In the current implementation, the `compute_distances_two_loops()` function calculates the L2 distance between each test point and each training point. The `engine/classifiers/k_nearest_neighbor.py` implementation can be adapted to use the L1 distance (also known as Manhattan or taxicab distance) instead of L2.

This requires computing the sum of the absolute differences between corresponding elements of the two points.


In [ ]:
# Recompute distances after switching the distance metric to L1 in the helper implementation.

# Validate the implementation:
dists_two = classifier.compute_distances_two_loops(X_test)
print(dists_two.shape)


In [ ]:
# Now run again the function predict_labels.
# We use k = 1 (which is Nearest Neighbor).
y_test_pred = classifier.predict_labels(dists_two, k=1)

# Compute and print the fraction of correctly predicted examples
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))


Expected observation: that k-NN using L1 distance is performing slightly better.


**Implementation note.**

Now that you’ve modified the `compute_distances_two_loops()` function to use L1 distance, reuse the above visualization function to visualize the top-10 closest images to a randomly chosen image from each class in the CIFAR-10 dataset, as you did previously with L2 distance.


In [ ]:
# Visualize the top-10 closest images to a randomly chosen image from each     #
# class in the CIFAR-10 test set. For each class, select one random test image #
# and find the 10 closest training images. Use the distances to color-code     #
# the retrieved images with a green border for correct labels and red for      #
# incorrect ones.                                                              #
classes = np.unique(y_train)

test_neighbors_dict = {}

# get neighbors
for i, class_id in enumerate(classes):
    class_indices = np.where(y_test == class_id)[0]
    random_idx = np.random.choice(class_indices)

    test_image = X_test[random_idx]
    test_label = y_test[random_idx]

    # Find the 10 nearest neighbors
    _, indices = classifier.kneighbors(test_image.reshape(1, -1), k=10)

    test_neighbors_dict[random_idx] = indices

# plot results
rows = 11
cols = 10
fig, axes = plt.subplots(rows, cols, figsize=(15, 15))

for i, (test_idx, neighbors) in enumerate(test_neighbors_dict.items()):
    # Plot the test image in the first row
    ax = axes[0, i]
    ax.imshow(X_test[test_idx].reshape((32, 32, 3)).astype('uint8'))
    ax.axis('off')
    ax.set_title(f'Class {y_test[test_idx]}')

    # Plot the nearest neighbors
    for j, neighbor_idx in enumerate(neighbors):
        ax = axes[j+1, i]
        ax.imshow(X_train[neighbor_idx].reshape((32, 32, 3)).astype('uint8'))
        ax.axis('off')

        # Check if the label of the neighbor is the same as the test samples label
        if y_train[neighbor_idx] == y_test[test_idx]:
            border_color = (0.5, 1, 0.5)
        else:
            border_color = 'red'

        border = patches.Rectangle((0, 0), 1, 1, linewidth=3, edgecolor=border_color, facecolor='none', transform=ax.transAxes)
        ax.add_patch(border)

plt.tight_layout()
plt.show()


**Concept check.**

After visualizing the top-10 closest images using both L2 and L1 distances, take a moment to compare the results:

•	When does L1 distance perform better than L2 distance?

•	Are there scenarios where L1 retrieves more semantically similar images than L2?

•	When does L2 distance outperform L1 distance?

•	Are there specific cases where L2 captures more morphologically similar images than L1?

Reflect on the following:

•	How do L1 and L2 distances emphasize different aspects of similarity between images (e.g., L1 might prioritize large pixel differences, while L2 emphasizes outliers due to squaring differences)?

•	Based on the qualitative results, which distance metric seems more appropriate for image retrieval in this context?

• What conclusions can we draw about the value of raw pixel values as features?

**Answer.** L1 distance is less sensitive to outliers thus is better for semantically similar images while L2 distance is better in capturing morphological similar pixels. But still both metrics in our model rely on poor representation. Overall, L2 is more appropriate when there is noise in the image.
